# 01 — RDD básico: procesamiento paralelo sobre `war_tweets.txt`

**Objetivo:** entender RDDs, la diferencia entre transformaciones (lazy) y acciones, y
comparar el paradigma MapReduce clásico con su equivalente en Spark.

**Dataset:** `war_tweets.txt` (~22.6 GB, JSON Lines) — ver `recursos/datasets/README.md`.
Cada línea es un tweet: `url`, `date`, `content`, `_type`, métricas de interacción.

Correr en un cluster real (Managed Service for Apache Spark / Dataproc) — este dataset no
cabe cómodamente en modo local. Ver `recursos/managed-spark-cluster/README.md`.

In [ ]:
from pyspark.sql import SparkSession
import json

spark = SparkSession.builder.appName("01_rdd_basico").getOrCreate()
sc = spark.sparkContext

RUTA = "gs://<TU-BUCKET>/raw/war_tweets/war_tweets.txt"

## 1. Cargar como RDD de texto plano

Cada línea del archivo es un objeto JSON independiente (formato *JSON Lines*) — por
eso se carga como texto y se parsea línea por línea, en vez de usar
`spark.read.json()` directamente (aunque esa también funciona; aquí se hace manual
a propósito para ver las transformaciones RDD paso a paso).

In [ ]:
rdd_crudo = sc.textFile(RUTA)
print(f"Líneas totales (acción -- dispara el conteo real): {rdd_crudo.count()}")
rdd_crudo.take(2)

## 2. Transformaciones (lazy: no se ejecutan hasta la siguiente acción)

Parsear cada línea a diccionario, quedarnos solo con `content`, y tokenizar en palabras.

In [ ]:
def parsear_tweet(linea):
    try:
        obj = json.loads(linea)
        return obj.get("content", "")
    except json.JSONDecodeError:
        return ""

contenidos = rdd_crudo.map(parsear_tweet).filter(lambda texto: texto != "")
palabras = contenidos.flatMap(lambda texto: texto.lower().split())
pares_palabra_uno = palabras.map(lambda palabra: (palabra, 1))

## 3. Acción: `reduceByKey` dispara la ejecución (aquí ocurre el shuffle)

Este es el mismo patrón MapReduce clásico (map → shuffle → reduce), pero en Spark
el resultado intermedio se mantiene en memoria entre pasos en vez de escribirse a
disco -- por eso es más rápido en pipelines con varias transformaciones encadenadas.

In [ ]:
conteo_palabras = pares_palabra_uno.reduceByKey(lambda a, b: a + b)

top_20 = conteo_palabras.takeOrdered(20, key=lambda par: -par[1])
print("=== Top 20 palabras más frecuentes ===")
for palabra, total in top_20:
    print(f"{palabra}: {total}")

## 4. Comparación conceptual con MapReduce clásico

En Hadoop MapReduce, este mismo cálculo requeriría:
- Un **Mapper** que emite `(palabra, 1)` por cada palabra de cada tweet
- Un **shuffle/sort** que agrupa por llave (lo gestiona el framework, escribiendo a disco entre fases)
- Un **Reducer** que suma los valores por llave

La diferencia clave: Spark mantiene los datos en memoria entre pasos (`map` → `flatMap`
→ `reduceByKey` no tocan disco entre sí), por eso es más rápido en pipelines iterativos
o con varias transformaciones encadenadas, como este.

## 5. Bonus: el JSON real no es plano — cuándo usar `read.json()` en vez de parsear a mano

`war_tweets.txt` tiene un esquema anidado de verdad (el tweet trae `user`, `quotedTweet`,
`inReplyToUser`, `media`, etc. como structs anidados varios niveles). El parseo manual de
arriba funcionó porque `content` sí es un campo de primer nivel — pero en cuanto se
necesita algo que SÍ está anidado (por ejemplo `user.username` o `user.followersCount`),
parsear a mano con `json.loads()` deja de ser práctico. Para eso existe
`spark.read.json()`: infiere el esquema completo (anidado) automáticamente.

In [ ]:
df_tweets = spark.read.json(RUTA)
df_tweets.printSchema()

Con el esquema inferido, los campos anidados se acceden con notación de punto —
igual que en la consulta de ejemplo que se corrió sobre este archivo:

In [ ]:
df_tweets.select("url", "date", "user.username", "user.followersCount") \
    .orderBy("date") \
    .show(2, truncate=False)

**Regla práctica para el curso:** si el archivo es JSON con estructura real (objetos
anidados, arrays de objetos), usar `read.json()` desde el inicio. El parseo manual con
RDDs (Secciones 1-3 de este notebook) tiene valor **pedagógico** — enseña qué es
map/shuffle/reduce por debajo — pero no es como se leería este archivo en un pipeline
real una vez que ya se entendió el concepto.

In [ ]:
spark.stop()